# Proyecto 1

## Diagnóstico inicial y plan de limpieza

Las 17 variables del conjunto: `CODIGO, DISTRITO, DEPARTAMENTO, MUNICIPIO, ESTABLECIMIENTO, DIRECCION, TELEFONO,
SUPERVISOR, DIRECTOR, NIVEL, SECTOR, AREA, STATUS, MODALIDAD, JORNADA, PLAN, DEPARTAMENTAL`.


In [26]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 60)

RAW = Path("data/diversificado_consolidado.csv")

## 1–2. Carga de los datos crudos (descargados y guardados en .csv)

In [27]:
def cargar(path):
    for enc in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            return pd.read_csv(path, dtype=str, keep_default_na=False, encoding=enc)
        except (UnicodeDecodeError, UnicodeError):
            continue
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding="latin-1")

df = cargar(RAW)
print(f"Registros: {len(df):,}   Variables: {df.shape[1]}")
df.head()

Registros: 11,867   Variables: 17


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
3,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,78232301,JOSE ARTURO CHOC CHEN,VIRGINIA SOLANO SERRANO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
4,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,MOISÉS ADRIÁN LÓPEZ PÉREZ,ESTELA DEL CARMEN QUIM ROSALES,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ


### Utilidades
Tratamos como **faltante** las celdas vacías y los marcadores de ausencia (`--`, `---`, `-`, `N/A`), que el portal usa en lugar de nulos reales.

In [28]:
PLACEHOLDERS = {"", "nan", "none", "null", "n/a", "-", "--", "---", "----", "s/n", "sin dato"}

def es_faltante(serie):
    s = serie.astype(str).str.strip().str.lower()
    return s.isin(PLACEHOLDERS)

def sin_tildes(s):
    s = unicodedata.normalize("NFKD", str(s))
    return "".join(ch for ch in s if not unicodedata.combining(ch))

def buscar_col(*claves):
    for col in df.columns:
        cl = sin_tildes(col).lower()
        if all(k in cl for k in claves):
            return col
    return None

faltante_mask = pd.DataFrame({col: es_faltante(df[col]) for col in df.columns})

## 3. Diagnóstico del estado inicial

### a) Número de registros y variables

In [29]:
print(f"Registros (filas): {df.shape[0]:,}")
print(f"Variables (columnas): {df.shape[1]}")
pd.DataFrame({"variable": df.columns})

Registros (filas): 11,867
Variables (columnas): 17


,variable
0,CODIGO
1,DISTRITO
2,DEPARTAMENTO
3,MUNICIPIO
4,ESTABLECIMIENTO
5,DIRECCION
6,TELEFONO
7,SUPERVISOR
8,DIRECTOR
9,NIVEL


### b) Tipo de dato de cada variable
Todo se cargó como texto para preservar ceros a la izquierda (códigos, teléfonos). Aquí se infiere además el **tipo semántico** observando el contenido.

In [30]:
def tipo_semantico(serie):
    v = serie[~es_faltante(serie)].astype(str).str.strip()
    if v.empty:
        return "vacía"
    if v.str.fullmatch(r"\d{2}-\d{2}-\d{4}-\d{2}").mean() > 0.8:
        return "código establecimiento (##-##-####-##)"
    if v.str.fullmatch(r"\d{2}-\d{2,3}(-\d{3,4})?").mean() > 0.6:
        return "código distrito"
    if v.str.fullmatch(r"[-+()\d\s/]{6,}").mean() > 0.7:
        return "teléfono"
    if v.nunique() <= max(30, 0.02 * len(v)):
        return "categórica"
    return "texto libre"

tipos = pd.DataFrame({
    "dtype_pandas": df.dtypes.astype(str),
    "tipo_semantico": {col: tipo_semantico(df[col]) for col in df.columns},
    "ejemplo": {col: next((x for x in df[col] if str(x).strip() not in PLACEHOLDERS), "") for col in df.columns},
})
tipos

,dtype_pandas,tipo_semantico,ejemplo
CODIGO,object,código establecimiento (##-##-####-##),16-01-0137-46
DISTRITO,object,código distrito,16-006
DEPARTAMENTO,object,categórica,ALTA VERAPAZ
MUNICIPIO,object,texto libre,COBAN
ESTABLECIMIENTO,object,texto libre,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN
DIRECCION,object,texto libre,6A. AVENIDA 1-15 ZONA 4
TELEFONO,object,teléfono,77945104
SUPERVISOR,object,texto libre,JORGE EDUARDO PAQUE LAZARO
DIRECTOR,object,texto libre,GUSTAVO ADOLFO SIERRA POP
NIVEL,object,categórica,DIVERSIFICADO


### c) Cantidad y porcentaje de valores faltantes por variable

In [31]:
faltantes = pd.DataFrame({
    "n_faltantes": faltante_mask.sum(),
    "pct_faltantes": (faltante_mask.mean() * 100).round(2),
}).sort_values("pct_faltantes", ascending=False)
faltantes

,n_faltantes,pct_faltantes
DIRECTOR,2024,17.06
TELEFONO,946,7.97
SUPERVISOR,535,4.51
DISTRITO,532,4.48
DIRECCION,80,0.67
ESTABLECIMIENTO,5,0.04
CODIGO,0,0.00
MUNICIPIO,0,0.00
DEPARTAMENTO,0,0.00
NIVEL,0,0.00


### d) Cantidad de valores únicos por variable

In [32]:
unicos = pd.DataFrame({
    "n_unicos": {col: df.loc[~es_faltante(df[col]), col].str.strip().nunique() for col in df.columns},
})
unicos["n_registros"] = len(df)
unicos["ratio_unicidad"] = (unicos["n_unicos"] / unicos["n_registros"]).round(3)
unicos.sort_values("n_unicos", ascending=False)

,n_unicos,n_registros,ratio_unicidad
CODIGO,11867,11867,1.000
DIRECCION,7437,11867,0.627
TELEFONO,6570,11867,0.554
ESTABLECIMIENTO,6312,11867,0.532
DIRECTOR,5512,11867,0.464
DISTRITO,1681,11867,0.142
SUPERVISOR,1280,11867,0.108
MUNICIPIO,352,11867,0.030
DEPARTAMENTAL,26,11867,0.002
DEPARTAMENTO,23,11867,0.002


### e) Registros duplicados exactos
Además del duplicado exacto (fila idéntica en las 17 columnas) reportamos el **duplicado por clave de negocio**: un mismo `CODIGO` puede repetirse legítimamente con distinta jornada/plan, así que se mide aparte.

In [33]:
col_cod = buscar_col("codigo")

dup_exactos = df.duplicated(keep="first").sum()
print(f"Filas duplicadas EXACTAS (todas las columnas): {dup_exactos}")

if col_cod:
    dup_cod = df[~es_faltante(df[col_cod])].duplicated(subset=[col_cod], keep=False).sum()
    print(f"Filas con CODIGO repetido (clave de negocio): {dup_cod}")
    clave = [c for c in [col_cod, buscar_col('jorn'), buscar_col('plan')] if c]
    dup_clave = df.duplicated(subset=clave, keep=False).sum()
    print(f"Filas con ({', '.join(clave)}) repetida: {dup_clave}")

Filas duplicadas EXACTAS (todas las columnas): 0
Filas con CODIGO repetido (clave de negocio): 0
Filas con (CODIGO, JORNADA, PLAN) repetida: 0


### f) Valores fuera de dominio o inconsistentes

Se comparan las variables categóricas contra su **dominio esperado** (departamentos oficiales, sectores/niveles/áreas/modalidades del portal). Se comparan en MAYÚSCULAS sin tildes.

In [34]:
DEPARTAMENTOS = {
    "ALTA VERAPAZ","BAJA VERAPAZ","CHIMALTENANGO","CHIQUIMULA","EL PROGRESO","ESCUINTLA",
    "GUATEMALA","HUEHUETENANGO","IZABAL","JALAPA","JUTIAPA","PETEN","QUETZALTENANGO",
    "QUICHE","RETALHULEU","SACATEPEQUEZ","SAN MARCOS","SANTA ROSA","SOLOLA",
    "SUCHITEPEQUEZ","TOTONICAPAN","ZACAPA",
}
SECTORES   = {"OFICIAL","PRIVADO","MUNICIPAL","COOPERATIVA"}
NIVELES    = {"DIVERSIFICADO"}
AREAS      = {"URBANA","RURAL"}
MODALIDADES= {"MONOLINGUE","BILINGUE"}

def norm(x):
    return sin_tildes(str(x)).upper().strip()

def fuera_de_dominio(col, dominio):
    v = df.loc[~es_faltante(df[col]), col].map(norm)
    return v[~v.isin(dominio)].value_counts()

for etiqueta, claves, dom in [
    ("DEPARTAMENTO", ("depart",),  DEPARTAMENTOS),   # nota: 'DEPARTAMENTAL' también matchea; ver abajo
    ("SECTOR",       ("sector",),  SECTORES),
    ("NIVEL",        ("nivel",),   NIVELES),
    ("AREA",         ("area",),    AREAS),
    ("MODALIDAD",    ("modalidad",),MODALIDADES),
]:
    col = etiqueta if etiqueta in df.columns else buscar_col(*claves)
    if col and col in df.columns:
        fuera = fuera_de_dominio(col, dom)
        print(f"\n### {col}: {len(fuera)} valor(es) fuera de dominio")
        print(fuera.to_string() if len(fuera) else "  (todos dentro de dominio)")


### DEPARTAMENTO: 1 valor(es) fuera de dominio
DEPARTAMENTO
CIUDAD CAPITAL    2161

### SECTOR: 0 valor(es) fuera de dominio
  (todos dentro de dominio)

### NIVEL: 0 valor(es) fuera de dominio
  (todos dentro de dominio)

### AREA: 1 valor(es) fuera de dominio
AREA
SIN ESPECIFICAR    3

### MODALIDAD: 0 valor(es) fuera de dominio
  (todos dentro de dominio)


In [35]:
# Categóricas sin dominio cerrado fijo: se listan sus valores para revisión
for col in [c for c in ["STATUS","JORNADA","PLAN"] if c in df.columns]:
    print(f"\n### {col} — valores observados:")
    print(df.loc[~es_faltante(df[col]), col].map(str.strip).value_counts().to_string())


### STATUS — valores observados:
STATUS
ABIERTA                    6860
CERRADA TEMPORALMENTE      3006
CERRADA DEFINITIVAMENTE    1841
TEMPORAL TITULOS            110
TEMPORAL NOMBRAMIENTO        50

### JORNADA — valores observados:
JORNADA
DOBLE          3772
VESPERTINA     3407
MATUTINA       3045
SIN JORNADA    1099
NOCTURNA        415
INTERMEDIA      129

### PLAN — valores observados:
PLAN
DIARIO(REGULAR)                          7452
FIN DE SEMANA                            2898
SEMIPRESENCIAL (FIN DE SEMANA)            542
SEMIPRESENCIAL (UN DÍA A LA SEMANA)       437
A DISTANCIA                               188
SEMIPRESENCIAL                            118
VIRTUAL A DISTANCIA                        70
SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)      67
SABATINO                                   59
DOMINICAL                                  27
MIXTO                                       4
IRREGULAR                                   3
INTERCALADO                                 2


In [36]:
# Consistencia entre columnas: DEPARTAMENTO vs DEPARTAMENTAL, y prefijo de DISTRITO vs código
if "DEPARTAMENTO" in df.columns and "DEPARTAMENTAL" in df.columns:
    difs = (df["DEPARTAMENTO"].map(norm) != df["DEPARTAMENTAL"].map(norm)).sum()
    print(f"Filas donde DEPARTAMENTO != DEPARTAMENTAL: {difs}  (si es 0, DEPARTAMENTAL es redundante)")

col_dist = buscar_col("distrito")
if col_cod and col_dist:
    pref_cod  = df[col_cod].str[:2]
    pref_dist = df.loc[~es_faltante(df[col_dist]), col_dist].str[:2]
    inconsist = (df.loc[pref_dist.index, col_cod].str[:2] != pref_dist).sum()
    print(f"Filas donde el prefijo de DISTRITO != prefijo de CODIGO: {inconsist}")

Filas donde DEPARTAMENTO != DEPARTAMENTAL: 4130  (si es 0, DEPARTAMENTAL es redundante)
Filas donde el prefijo de DISTRITO != prefijo de CODIGO: 1958


### g) Formatos inconsistentes

Se detectan: espacios dobles/sobrantes, **mojibake** (caracteres corruptos como `¿`, `Ã`, comillas escapadas `\'`), formato irregular del `DISTRITO`, teléfonos con varios números o separadores, y **fechas incrustadas** en direcciones.

In [37]:
def con_espacios(s):   return (s.astype(str) != s.astype(str).str.strip()) | s.astype(str).str.contains(r"\s{2,}", regex=True, na=False)
def con_mojibake(s):   return s.astype(str).str.contains(r"[¿�]|Ã.|â€|Â.|\\['\"]", regex=True, na=False)
def con_fecha(s):      return s.astype(str).str.contains(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", regex=True, na=False)

fmt = pd.DataFrame({
    "espacios_extra":   {c: int(con_espacios(df[c]).sum())  for c in df.columns},
    "posible_mojibake": {c: int(con_mojibake(df[c]).sum())  for c in df.columns},
    "fecha_incrustada": {c: int(con_fecha(df[c]).sum())     for c in df.columns},
})
print(fmt[(fmt.T != 0).any()].to_string())

# Formato de CODIGO y DISTRITO
if col_cod:
    mal_cod = (~df[col_cod].str.match(r"^\d{2}-\d{2}-\d{4}-\d{2}$", na=False)).sum()
    print(f"\nCODIGO con formato != ##-##-####-##: {mal_cod}")
if col_dist:
    d = df.loc[~es_faltante(df[col_dist]), col_dist]
    print("Formatos de DISTRITO (conteo por patrón de longitud):")
    print(d.str.replace(r'\d', '#', regex=True).value_counts().to_string())

col_tel = buscar_col("telefono")
if col_tel:
    t = df.loc[~es_faltante(df[col_tel]), col_tel]
    n_tel_raro = (~t.str.fullmatch(r"\d{8}")).sum()
    print(f"\nTELEFONO con separadores o múltiples números (no 8 dígitos limpios): {n_tel_raro} de {len(t)}")

                 espacios_extra  posible_mojibake  fecha_incrustada
ESTABLECIMIENTO               0                 2                 0
DIRECCION                     0                 5                18
TELEFONO                      0                 0                 1
DIRECTOR                      0                 1                 0

CODIGO con formato != ##-##-####-##: 0
Formatos de DISTRITO (conteo por patrón de longitud):
DISTRITO
##-##-####    6226
##-###        5039
##-             70

TELEFONO con separadores o múltiples números (no 8 dígitos limpios): 251 de 10921


### h) Identificación de problemas potenciales de calidad — resumen

Consolidación por variable de todo lo anterior.

In [38]:
resumen = pd.DataFrame(index=df.columns)
resumen["tipo"]             = tipos["tipo_semantico"]
resumen["pct_faltantes"]    = faltantes["pct_faltantes"]
resumen["n_unicos"]         = unicos["n_unicos"]
resumen["espacios_extra"]   = fmt["espacios_extra"]
resumen["posible_mojibake"] = fmt["posible_mojibake"]
resumen["fecha_incrustada"] = fmt["fecha_incrustada"]
resumen

,tipo,pct_faltantes,n_unicos,espacios_extra,posible_mojibake,fecha_incrustada
CODIGO,código establecimiento (##-##-####-##),0.00,11867,0,0,0
DISTRITO,código distrito,4.48,1681,0,0,0
DEPARTAMENTO,categórica,0.00,23,0,0,0
MUNICIPIO,texto libre,0.00,352,0,0,0
ESTABLECIMIENTO,texto libre,0.04,6312,0,2,0
DIRECCION,texto libre,0.67,7437,0,5,18
TELEFONO,teléfono,7.97,6570,0,0,1
SUPERVISOR,texto libre,4.51,1280,0,0,0
DIRECTOR,texto libre,17.06,5512,0,1,0
NIVEL,categórica,0.00,1,0,0,0


**Hallazgos principales (resumen cualitativo):**

- `NIVEL` es constante (`DIVERSIFICADO`) por el filtro de la descarga: no aporta variabilidad.
- `TELEFONO`, `DIRECTOR`, `DISTRITO` y `DIRECCION` concentran los **faltantes**; `DIRECTOR` además usa marcadores `--`/`---`.
- `TELEFONO` mezcla formatos: 8 dígitos, con guiones, y hasta **dos números** en una celda.
- `DISTRITO` tiene **varios formatos** distintos.
- Textos (`ESTABLECIMIENTO`, `DIRECCION`, `SUPERVISOR`, `DIRECTOR`) con **espacios dobles, mojibake** (`¿`, comillas escapadas) y **tildes inconsistentes** (`SURÁM`/`SURAM`).
- Alguna `DIRECCION` trae una **fecha incrustada** (p. ej. `02/01/2018`), dato que no corresponde a la variable.
- `AREA` puede traer valores fuera de `{URBANA, RURAL}` (p. ej. `SIN ESPECIFICAR`).


## 4. Plan de limpieza

Para cada variable: **problema** encontrado, **regla** de corrección (y por qué funcionaría) y **riesgo** de la transformación. Es la guía que usará el equipo para ejecutar la limpieza después.

| Variable | Problema encontrado | Regla de corrección (y por qué funciona) | Riesgo |
|---|---|---|---|
| `DISTRITO` | Dos formatos (`16-006` y `16-01-0926`) y algunos faltantes. | Documentar ambos formatos; separar en “distrito corto/largo” solo si el análisis lo requiere; no imputar faltantes. | Medio: forzar un único formato podría destruir información; mejor dejar como texto y anotar. |
| `DEPARTAMENTO` | Mayúsculas/tildes; validar contra los 22 oficiales. | Normalizar a MAYÚSCULAS sin tildes y mapear al catálogo. Dominio cerrado y conocido. | Bajo. |
| `MUNICIPIO` | Tildes/espacios inconsistentes. | Normalizar (MAYÚSCULAS sin tildes, colapsar espacios) y validar contra municipios del departamento. | Medio: un municipio mal escrito podría no mapear; se deja marcado para revisión. |
| `ESTABLECIMIENTO` | Espacios dobles, comillas escapadas, mojibake, algún vacío. | `strip` + colapsar espacios; corregir codificación (`ftfy`); dejar vacío como faltante. | Medio: la corrección de mojibake mal aplicada puede alterar nombres correctos; revisar muestra. |
| `DIRECCION` | Espacios, mojibake y **fechas incrustadas** (`02/01/2018`). | Colapsar espacios; extraer/eliminar la fecha con regex `\d{1,2}/\d{1,2}/\d{2,4}` moviéndola a una nota. | Medio: una fecha podría ser parte legítima de la dirección (raro); revisar los casos detectados. |
| `TELEFONO` | Vacíos, guiones, y **dos números** en una celda. | Quedarse con dígitos; si hay 16 dígitos, separar en dos columnas; validar longitud 8 (GT); vacío = faltante. | Bajo/medio: números atípicos se marcan como inválidos, **no** se borran. |
| `SUPERVISOR` | Tildes/espacios inconsistentes. | `strip` + colapsar espacios; corregir codificación. No imputar. | Bajo. |
| `DIRECTOR` | Marcadores `--`/`---`/`-` y vacíos. | Convertir marcadores a faltante (NaN); `strip`; corregir codificación. | Bajo. |
| `NIVEL` | Constante `DIVERSIFICADO`. | Verificar que todo sea `DIVERSIFICADO`; se puede conservar como control o descartar por nula varianza. | Bajo. |
| `SECTOR` | Dominio `{OFICIAL, PRIVADO, MUNICIPAL, COOPERATIVA}`. | Estandarizar contra el dominio cerrado. | Bajo. |
| `AREA` | Puede traer `SIN ESPECIFICAR` fuera de `{URBANA, RURAL}`. | Estandarizar; `SIN ESPECIFICAR` → categoría explícita o faltante, según criterio. | Bajo. |
| `STATUS` | Varias categorías (`ABIERTA`, `CERRADA…`, `TEMPORAL NOMBRAMIENTO`). | Estandarizar contra el conjunto observado; unificar variantes de escritura. | Bajo. |
| `MODALIDAD` | Dominio `{MONOLINGUE, BILINGUE}`. | Estandarizar contra el dominio. | Bajo. |
| `JORNADA` | `SIN JORNADA` y variantes. | Estandarizar; decidir si `SIN JORNADA` es faltante o categoría propia. | Bajo. |
| `PLAN` | Muchas variantes (`SEMIPRESENCIAL (FIN DE SEMANA)`, `VIRTUAL A DISTANCIA`, …). | Definir catálogo de planes y mapear variantes a categorías canónicas. | Medio: agrupar de más puede perder matices; documentar el mapeo. |
| `DEPARTAMENTAL` | Parece **redundante** con `DEPARTAMENTO`. | Confirmar igualdad fila a fila (inciso f); si coincide siempre, eliminar la columna. | Bajo, si se confirma la redundancia. |
| *Filas* | Duplicados exactos y por clave de negocio (mismo `CODIGO`, distinta jornada/plan). | Eliminar solo **duplicados exactos** con `drop_duplicates()`; los de clave de negocio se conservan (son servicios distintos). | Bajo: deduplicar por fila completa evita borrar registros legítimos. |
